<a href="https://colab.research.google.com/github/RainanCoelho/Projeto-Analise-Exploratoria-Grupo8-Gestao_de_Vendedores-Parceiros/blob/main/Dataset_olist_products_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**FRENTE 1**

In [2]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/RainanCoelho/Projeto-Analise-Exploratoria-Grupo8-Gestao_de_Vendedores-Parceiros/refs/heads/main/olist_products_dataset.csv"
df_opd = pd.read_csv(URL)
df_opd.columns = ["product_id","product_category_name","product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]
df_opd.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


PARTE 1: CONHECIMENTO DOS DADOS

In [ ]:
df_opd.shape #Mostra no console a quantidade de linhas e colunas do dataset

In [ ]:
df_opd.info() #Mostra o tipo e a quantidade de valores não nulos

In [ ]:
df_opd.describe() #Mostra a estatística das tabelas, como média, valor máximo, mínimo, valores interquartis e mais

In [ ]:
df_opd.dtypes #Mostra os tipos dos valores de cada coluna

PARTE 2: CORREÇÃO DOS TIPOS

In [ ]:
def corrigir_tipos_colunas(df):
  df = df.copy()
  coluna_corrigir = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
  for coluna in coluna_corrigir:
    df[coluna] = df[coluna].astype("Int64")
  return df

df_opd = corrigir_tipos_colunas(df_opd)
df_opd.dtypes

**EXPLICAÇÃO**: Altera o tipo dos dados das colunas: "product_name_lenght", "product_description_lenght" e "product_photos_qty", já que elas estavam marcando como float, mas faz mais sentido elas serem lidas do tipo inteiro

PARTE 3: Padronização de nomes

In [ ]:
def padronizar_categoria(df):
  df = df.copy()
  df["product_category_name"] = df["product_category_name"].str.strip().str.lower()
  return df

df_opd = padronizar_categoria(df_opd)

**EXPLICAÇÃO**: Essa função serve para padronizar os nomes da coluna: "product_category_name", para que não haja erro na leitura dos nomes

PARTE 4: VERIFICAÇÃO DE DUPLICADAS E ID

In [ ]:
df_opd.isnull().sum() #Verifica a quantidade de valores nulos por coluna

**RESPOSTA**: Foi verificado 610 valores nulos nas colunas: product_category_name,	product_name_lenght,	product_description_lenght e product_photos_qty, e 2 valores nulos nas colunas: product_weight_g,	product_length_cm,	product_height_cm e	product_width_cm e foi verificado que e a coluna product_id não tem nenhum valor duplicado

In [ ]:
print("product_id duplicado:", df_opd["product_id"].duplicated().sum())
print("product_id nulo:", df_opd["product_id"].isnull().sum())

product_id duplicado: 0
product_id nulo: 0


**EXPLICAÇÃO**: Esse comando serve para ver se existe algum valor duplicado ou nulo na coluna "product_id"

In [ ]:
colunas = ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
print("Linhas com as 4 colunas vazias:", df_opd[colunas].isnull().all(axis=1).sum())

Linhas com as 4 colunas vazias: 610


**EXPLICAÇÃO**: Esse comando busca entender se os 610 valores nulos pertencem a um mesmo produto, ou seja, existem 610 linhas onde os valores das colunas estão nulas na mesma linha

In [ ]:
medidas = ["product_weight_g", "product_length_cm", "product_height_cm","product_width_cm"]
print("Linhas com 4 medidas vazias:", df_opd[medidas].isnull().all(axis=1).sum())

Linhas com 4 medidas vazias: 2


**EXPLICAÇÃO**: Esse comando busca entender se os 2 valores nulos pertencem a um mesmo produto, ou seja, existem 2 linhas onde os valores das colunas estão nulas na mesma linha

In [ ]:
df_opd.duplicated().sum() #Verifica se existe algum valor duplicado na tabela

np.int64(0)

**RESPOSTA**: Não foi verificado nenhuma linha duplicada na tabela

PARTE 5: VERIFICAÇÃO DE VALORES IMPOSSÍVEIS

In [ ]:
colunas= ["product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]
for coluna in colunas:
  print((df_opd[coluna] <= 0).sum())

0
0
0
4
0
0
0


**EXPLICAÇÃO**: Esse comando busca ver se nas tabelas de valor numérico existe algum valor nulo ou abaixo de 0

In [ ]:
def anular_peso_zero(df):
  df = df.copy()
  df.loc[df["product_weight_g"] == 0, "product_weight_g"] = np.nan
  return df

print("Antes:", (df_opd["product_weight_g"] == 0).sum())
df_opd = anular_peso_zero(df_opd)
print("Depois:", (df_opd["product_weight_g"] == 0).sum())

Antes: 4
Depois: 0


**EXPLICAÇÃO**: Os 4 valores que estão exatamente iguais a 0 na coluna "product_weight_g" foram substituidos como valores nulos, mas não excluídos, já que a exclusão dos mesmos iria fazer uma intervenção mínima no sistema

PARTE 6: Verificação de valores Outlier

In [ ]:
def verificar_outliers(df):
  colunas = ["product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]
  for coluna in colunas:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    outlier = df[(df[coluna] > limite_sup) | (df[coluna] < limite_inf)]
    acima = df[df[coluna] > limite_sup]
    abaixo = df[df[coluna] < limite_inf]
    print(coluna, "| acima:", len(acima), "| abaixo:", len(abaixo))


verificar_outliers(df_opd)

product_name_lenght | acima: 0 | abaixo: 290
product_description_lenght | acima: 2078 | abaixo: 0
product_photos_qty | acima: 849 | abaixo: 0
product_weight_g | acima: 4551 | abaixo: 0
product_length_cm | acima: 1380 | abaixo: 0
product_height_cm | acima: 1892 | abaixo: 0
product_width_cm | acima: 912 | abaixo: 0


**EXPLICAÇÃO**: Com base nos parâmetros IQR,  foi notado que em todas as colunas de peso e medidas: 4.551 produtos acima de 4.300 g, 1.380 acima de 68 cm de comprimento, 1.892 acima de 40,5 cm de altura e 912 acima de 52,5 cm de largura. E os valores que foram identificados como outliers, eles podem ser facilmente aceitos levando em conta a diferença de produtos cadastrados, então uma exclusão de dados outliers não seria a melhor opção